### **Testing Architectures**

In [1]:
from src.umich_slm import UMichSLMConfig, UMichSLMForCausalLM
import torch
from src.umich_slm.configs.presets import estimate_parameters

/Users/omertafveez/.virtualenvs/DL/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
config = UMichSLMConfig(
    hidden_size= 576,
    intermediate_size= 1536,
    num_hidden_layers= 28,
    num_attention_heads= 9,
    num_key_value_heads= 3,
    max_position_embeddings= 2048,
    tie_word_embeddings= True,
    use_block_weight_sharing= False,
    weight_sharing_block_size= 2,
    vocab_size= 32000,
)
# backbone ≈ 97.6M


In [16]:
model = UMichSLMForCausalLM(config)
params = [(n, p.numel()) for n, p in model.named_parameters()]

embed_params = sum(p for n, p in params if "embed_tokens" in n)
lm_head_params = sum(p for n, p in params if "lm_head" in n)
vocab_params = embed_params + lm_head_params
backbone_params = sum(p for n, p in params if "embed_tokens" not in n and "lm_head" not in n)
total_params = sum(p for _, p in params)


# in Millions
print(f"Backbone:   {backbone_params/1e6:.2f}M")
print(f"Vocab/Head: {vocab_params/1e6:.2f}M")
print(f"Total:      {total_params/1e6:.2f}M")

Backbone:   99.12M
Vocab/Head: 18.43M
Total:      117.56M


In [4]:
input_ids = torch.randint(0, config.vocab_size, (1, 16))
with torch.no_grad():
    out = model(input_ids=input_ids)

In [5]:
print("logits shape:", out.logits.shape)  # expect (1, 16, vocab_size)


logits shape: torch.Size([1, 16, 49152])


In [6]:
with torch.no_grad():
    out_cached = model(input_ids=input_ids, use_cache=True)

print("cache type:", type(out_cached.past_key_values))

cache type: <class 'src.umich_slm.modules.griffin.GriffinDynamicCache'>
